In [0]:
from pyspark.sql import functions as F

SILVER = "workspace.s4lake_silver"
BRONZE = "workspace.s4lake_bronze"

vbrk = spark.table(f"{BRONZE}.vbrk")
vbrp = spark.table(f"{BRONZE}.vbrp")

ordens = spark.table(f"{SILVER}.ordens_venda")

# Cabeçalho: uma linha por fatura
faturas = vbrk.select(
    F.col("VBELN").alias("cod_fatura"),
    F.col("FKART").alias("tipo_fatura"),
    F.to_date("FKDAT", "yyyyMMdd").alias("data_fatura"),
    F.col("KUNAG").alias("cod_cliente"),
    F.col("BUKRS").alias("empresa"),
    F.col("NETWR").cast("decimal(15,2)").alias("valor_liquido"),
    F.col("MWSBK").cast("decimal(15,2)").alias("valor_imposto"),
    F.col("WAERK").alias("moeda"),
    F.col("ZTERM").alias("cond_pagamento"),
)

# Itens: uma linha por item por fatura
faturas_itens = (
    vbrp.alias("i")
    .join(
        faturas.select("cod_fatura").alias("o"),
        F.col("i.VBELN") == F.col("o.cod_fatura"),
        "left"
    )
    .join(
        ordens.select("cod_ordem").alias("ov"),
        F.col("i.AUBEL") == F.col("ov.cod_ordem"),
        "left"
    )
    .select(
        F.col("i.VBELN").alias("cod_fatura"),
        F.col("i.POSNR").alias("item"),
        F.col("i.AUBEL").alias("cod_ordem_origem"),
        F.col("i.AUPOS").alias("item_ordem_origem"),
        F.col("i.MATNR").alias("cod_material"),
        F.col("i.FKIMG").cast("decimal(15,3)").alias("quantidade_faturada"),
        F.col("i.VRKME").alias("unidade"),
        F.col("i.NETWR").cast("decimal(15,2)").alias("valor_liquido"),
        F.col("o.cod_fatura").isNotNull().alias("_fatura_existe"),
        F.col("ov.cod_ordem").isNotNull().alias("_ordem_existe"),
    )
    .withColumn(
        "motivo_rejeicao",
        F.when(~F.col("_fatura_existe"), "fatura inexistente na VBRK")
         .when(~F.col("_ordem_existe"), "ordem de venda de origem inexistente na VBAK")
         .when(F.col("quantidade_faturada").isNull() | (F.col("quantidade_faturada") <= 0), "quantidade zerada ou negativa")
         .when(F.col("valor_liquido") < 0, "valor negativo")
    )
    .drop("_fatura_existe", "_ordem_existe")
)

def salvar(df, tabela):
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{SILVER}.{tabela}")

salvar(faturas, "faturas_venda")
salvar(faturas_itens.filter("motivo_rejeicao IS NULL").drop("motivo_rejeicao"), "faturas_venda_itens")
salvar(
    faturas_itens.filter("motivo_rejeicao IS NOT NULL").withColumn("_quarentena_em", F.current_timestamp()),
    "quarentena_faturas_venda_itens",
)

In [0]:
SILVER = "workspace.s4lake_silver"

spark.sql(f"""
    SELECT count(*) AS faturas
    FROM {SILVER}.faturas_venda
""").show()

spark.sql(f"""
    SELECT count(*) AS item
    FROM {SILVER}.faturas_venda_itens
""").show()

spark.sql(f"""
    SELECT count(*) AS itens_quarentena
    FROM {SILVER}.quarentena_faturas_venda_itens
""").show()

spark.sql(f"""
    SELECT motivo_rejeicao, count(*) AS itens
    FROM {SILVER}.quarentena_faturas_venda_itens
    GROUP BY motivo_rejeicao
""").show(truncate=False)